# 🔄 Independent Training + Progressive Averaging

**Стратегия:**
- Каждый день обучается **независимо** (модель создается с нуля)
- После каждого дня веса **усредняются**
- Усредненная модель **тестируется** на 2025-09-30
- Наблюдаем **прогрессивное улучшение** с добавлением дней

**Преимущества:**
- ✅ Нет Catastrophic Forgetting
- ✅ Каждый день равноценен
- ✅ Усреднение улучшает обобщение

In [ ]:
# Импорт необходимых библиотек
import sys
import os
from pathlib import Path

# Добавляем src в путь для импорта модулей
sys.path.append(str(Path.cwd() / "src"))

import torch
import numpy as np
import pandas as pd
from copy import deepcopy

# Импорт наших модулей
from config import Config
from load_data import load_data
from tools import build_barrier_labels
from model import create_model, create_optimizer_and_scheduler
from dataset import BalancedBatchSampler, BalancedBatchBatchSampler, LazyWindowDataset
from torch.utils.data import DataLoader
from trainer import create_trainer

print(f"PyTorch версия: {torch.__version__}")
print(f"CUDA доступна: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA версия: {torch.version.cuda}")
print(f"Устройство: {'cuda' if torch.cuda.is_available() else 'cpu'}")

In [ ]:
# Создание конфигурации
config = Config.default()

# Настройка параметров
config.data.train_dates = ["2025-09-22", "2025-09-23", "2025-09-25", "2025-09-26"]
config.data.val_date = "2025-09-29"
config.data.test_date = "2025-09-30"
config.data.data_folder = "./data/npy"

config.model.tick_size = 1.0
config.model.theta_ticks = 5
config.model.horizon_sec = 2.0
config.model.window_length = 240
config.model.use_cost_sensitive_focal = True

config.training.epochs = 20

print("="*70)
print("📋 КОНФИГУРАЦИЯ INDEPENDENT TRAINING")
print("="*70)
print(f"\nДни обучения: {config.data.train_dates}")
print(f"День валидации: {config.data.val_date}")
print(f"День тестирования: {config.data.test_date}")
print(f"Эпохи на день: {config.training.epochs}")
print(f"\nСтратегия: Independent Training + Progressive Averaging")
print(f"Тестирование после каждого дня на: {config.data.test_date}\n")

In [ ]:
# Загрузка данных для обучения
print("📥 Загрузка данных для обучения...")
print("⚡ Используется LazyWindowDataset - экономия памяти!\n")

train_data_loaders = []

for i, train_date in enumerate(config.data.train_dates):
    print(f"День {i+1}/{len(config.data.train_dates)}: {train_date}")
    
    features_file = config.data.get_features_file(train_date)
    prices_file = config.data.get_prices_file(train_date)
    X, ms, mid = load_data(features_file, prices_file)
    
    y_all = build_barrier_labels(
        ms, mid, 
        tick_size=config.model.tick_size,
        theta_ticks=config.model.theta_ticks, 
        horizon_sec=config.model.horizon_sec
    )
    
    valid = (y_all != -1)
    Xv, yv = X[valid], y_all[valid]
    
    train_ds = LazyWindowDataset(Xv, yv, config.model.window_length)
    y_win = yv[config.model.window_length-1:]
    
    print(f"  Классы: {np.unique(y_win, return_counts=True)}")
    print(f"  Окна: {len(train_ds)}\n")
    
    base_sampler = BalancedBatchSampler(
        y_win, 
        batch_size=config.training.batch_size, 
        num_classes=config.model.num_classes, 
        seed=config.training.seed
    )
    batch_sampler = BalancedBatchBatchSampler(base_sampler, config.training.batch_size)
    train_dl = DataLoader(train_ds, batch_sampler=batch_sampler, num_workers=0)
    train_data_loaders.append(train_dl)

print(f"✅ Загружено {len(train_data_loaders)} дней для обучения")

In [ ]:
# Загрузка валидационных и тестовых данных
print(f"\n📥 Загрузка валидации: {config.data.val_date}")
val_features_file = config.data.get_features_file(config.data.val_date)
val_prices_file = config.data.get_prices_file(config.data.val_date)
X_val, ms_val, mid_val = load_data(val_features_file, val_prices_file)

y_val_all = build_barrier_labels(
    ms_val, mid_val, 
    tick_size=config.model.tick_size,
    theta_ticks=config.model.theta_ticks, 
    horizon_sec=config.model.horizon_sec
)

valid_val = (y_val_all != -1)
Xv_val, yv_val = X_val[valid_val], y_val_all[valid_val]
val_ds = LazyWindowDataset(Xv_val, yv_val, config.model.window_length)
val_dl = DataLoader(val_ds, batch_size=config.training.batch_size, shuffle=False, num_workers=0)
print(f"  Окна: {len(val_ds)}")

print(f"\n📥 Загрузка теста: {config.data.test_date}")
test_features_file = config.data.get_features_file(config.data.test_date)
test_prices_file = config.data.get_prices_file(config.data.test_date)
X_test, ms_test, mid_test = load_data(test_features_file, test_prices_file)

y_test_all = build_barrier_labels(
    ms_test, mid_test, 
    tick_size=config.model.tick_size,
    theta_ticks=config.model.theta_ticks, 
    horizon_sec=config.model.horizon_sec
)

valid_test = (y_test_all != -1)
Xv_test, yv_test = X_test[valid_test], y_test_all[valid_test]
test_ds = LazyWindowDataset(Xv_test, yv_test, config.model.window_length)
test_dl = DataLoader(test_ds, batch_size=config.training.batch_size, shuffle=False, num_workers=0)
print(f"  Окна: {len(test_ds)}")

print(f"\n✅ Все данные загружены")

In [ ]:
# Вспомогательные функции

def average_weights(state_dicts):
    """Усреднить веса нескольких моделей"""
    avg_state = {}
    num_models = len(state_dicts)
    
    for key in state_dicts[0].keys():
        avg_state[key] = sum(sd[key].float() for sd in state_dicts) / num_models
    
    return avg_state

def evaluate_and_display(model, test_loader, device, title='Оценка модели'):
    """Оценить модель и красиво вывести результаты"""
    model.eval()
    total_loss = 0.0
    total_samples = 0
    y_true, y_pred = [], []
    
    with torch.no_grad():
        for xb, yb in test_loader:
            xb, yb = xb.to(device), yb.to(device)
            logits = model(xb)
            loss = torch.nn.functional.cross_entropy(logits, yb, reduction='mean')
            total_loss += loss.item() * xb.size(0)
            total_samples += xb.size(0)
            y_true.append(yb.cpu().numpy())
            y_pred.append(logits.argmax(1).cpu().numpy())
    
    y_true = np.concatenate(y_true)
    y_pred = np.concatenate(y_pred)
    
    from sklearn.metrics import f1_score, confusion_matrix
    
    test_loss = total_loss / total_samples
    macro_f1 = f1_score(y_true, y_pred, average='macro')
    f1_scores = f1_score(y_true, y_pred, labels=[0, 1, 2], average=None)
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1, 2])
    
    print(f"\n{title}:")
    print(f"  Loss: {test_loss:.4f}")
    print(f"  Macro F1: {macro_f1:.4f}")
    print(f"  F1 ↓: {f1_scores[0]:.3f} | F1 ○: {f1_scores[1]:.3f} | F1 ↑: {f1_scores[2]:.3f}")
    print(f"  Confusion Matrix:")
    print(f"  {cm}")
    
    return {
        'loss': test_loss,
        'macro_f1': macro_f1,
        'f1_down': f1_scores[0],
        'f1_flat': f1_scores[1],
        'f1_up': f1_scores[2],
        'confusion_matrix': cm
    }

print("✅ Вспомогательные функции определены")

In [ ]:
# Тренировка для одного дня
TRAIN_DATE = "2025-09-26" 
SUB_PATH = 1
config.training.epochs = 10

if TRAIN_DATE not in config.data.train_dates:
    print(f"❌ Ошибка: Дата {TRAIN_DATE} не найдена!")
    print(f"   Доступные даты: {config.data.train_dates}")

day_idx = config.data.train_dates.index(TRAIN_DATE)
day_num = day_idx + 1

# Получить DataLoader для этого дня
train_loader = train_data_loaders[day_idx]
    
# ✨ СОЗДАЕМ НОВУЮ МОДЕЛЬ (с нуля!)
model_day, criterion_day = create_model(config.model)
optimizer_day, scheduler_day, scaler_day = create_optimizer_and_scheduler(model_day, config.training)
model_day = model_day.to(config.training.device)

# Создаем тренера для этого дня
trainer_day = create_trainer(
    model=model_day,
    criterion=criterion_day,
    optimizer=optimizer_day,
    scheduler=scheduler_day,
    scaler=scaler_day,
    device=config.training.device,
    grad_clip_norm=config.training.grad_clip_norm
)

# загрузка весов модели
model_file = f"./models_save/{SUB_PATH}/model_day_{TRAIN_DATE}.pt"  # Можно изменить на любой файл: "model_day_2.pt", "best_deeplob_like.pt" и т.д.

print(f"📥 Загрузка модели из файла: {model_file}")
print(f"   Путь: {os.path.abspath(model_file)}")

if os.path.exists(model_file):
    # Загрузка весов модели
    state_dict = torch.load(model_file, map_location=config.training.device)
    model_day.load_state_dict(state_dict)

    print("✅ Модель успешно загружена!")
else:
    print(f"❌ ОШИБКА: Файл '{os.path.abspath(model_file)}' не найден!")

    
# Обучение
day_results = trainer_day.train(
    train_loader=train_loader,
    val_loader=val_dl,
    epochs=config.training.epochs,
    verbose=True
)

# Сохраняем модель дня
model_file = f"./models_save/{SUB_PATH+1}/model_day_{TRAIN_DATE}.pt"  
torch.save(model_day.state_dict(), model_file)
print(f"\n💾 Модель дня сохранена: {model_file}")
print(f"📊 Лучший macro-F1 (val): {day_results['best_score']:.4f}")

🔧 Создание улучшенной модели:
   input_features: 211
   hidden_size: 96
   num_classes: 3
   Residual connections: ✓
   Dilations: [1, 2, 4, 8, 16]
   Attention mechanism: ✓
📉 Используется Weighted CrossEntropyLoss
   Веса классов: [10.0, 1.0, 10.0]
   Label smoothing: 0.1
📥 Загрузка модели из файла: ./models_save/1/model_day_2025-09-26.pt
   Путь: D:\python-prj\orderBookTrain\models_save\1\model_day_2025-09-26.pt
✅ Модель успешно загружена!

🚀 ИНФОРМАЦИЯ ОБ ОБУЧЕНИИ

📊 МОДЕЛЬ:
   Всего параметров: 535,395
   Обучаемых параметров: 535,395
   Размер модели: ~2.04 МБ (float32)

📦 ДАННЫЕ:
   Train батчей: 283
   Val батчей: 274
   Размер батча: torch.Size([512, 211, 240])
   Тип данных: torch.float32
   Устройство: cpu
   Классы в батче: [0, 1, 2]
   Распределение: [171, 171, 170]

⚙️  ОПТИМИЗАТОР:
   Тип: AdamW
   Learning rate: 1.00e-04
   Weight decay: 5.00e-04
   Scheduler: CosineAnnealingLR
   Gradient clipping: 0.3

📉 LOSS FUNCTION:
   Тип: WeightedCrossEntropyLoss
   label_smoothin

In [ ]:
# ============================================================
# INDEPENDENT TRAINING + PROGRESSIVE AVERAGING
# ============================================================

print("="*70)
print("🚀 НАЧАЛО INDEPENDENT TRAINING + PROGRESSIVE AVERAGING")
print("="*70)
print("\nСтратегия: Каждый день обучается независимо")
print("После каждого дня веса усредняются и тестируются\n")

# Хранилище моделей и результатов
all_day_models = []  # Веса моделей
all_day_scores = []  # F1 на валидации
progressive_test_results = []  # Результаты на тесте после усреднения

for day_idx, train_loader in enumerate(train_data_loaders):
    day_num = day_idx + 1
    train_date = config.data.train_dates[day_idx]
    
    print(f"\n{'='*70}")
    print(f"📅 ДЕНЬ {day_num}/4: {train_date} (НЕЗАВИСИМОЕ ОБУЧЕНИЕ)")
    print(f"{'='*70}\n")
    
    # ✨ СОЗДАЕМ НОВУЮ МОДЕЛЬ (с нуля!)
    model_day, criterion_day = create_model(config.model)
    optimizer_day, scheduler_day, scaler_day = create_optimizer_and_scheduler(model_day, config.training)
    model_day = model_day.to(config.training.device)
    
    print(f"✨ Создана новая модель с нуля")
    print(f"   Параметров: {sum(p.numel() for p in model_day.parameters()):,}\n")
    
    # Создаем тренера для этого дня
    trainer_day = create_trainer(
        model=model_day,
        criterion=criterion_day,
        optimizer=optimizer_day,
        scheduler=scheduler_day,
        scaler=scaler_day,
        device=config.training.device,
        grad_clip_norm=config.training.grad_clip_norm
    )

    # загрузка весов модели
    model_file = f"./models_save/1/model_day_{day_num}.pt"  # Можно изменить на любой файл: "model_day_2.pt", "best_deeplob_like.pt" и т.д.

    print(f"📥 Загрузка модели из файла: {model_file}")
    print(f"   Путь: {os.path.abspath(model_file)}")

    if os.path.exists(model_file):
        # Загрузка весов модели
        state_dict = torch.load(model_file, map_location=config.training.device)
        model_day.load_state_dict(state_dict)
    
        print("✅ Модель успешно загружена!")
    else:
        print(f"❌ ОШИБКА: Файл '{os.path.abspath(model_file)}' не найден!")

    
    # Обучение
    day_results = trainer_day.train(
        train_loader=train_loader,
        val_loader=val_dl,
        epochs=config.training.epochs,
        verbose=True
    )
    
    # Сохраняем веса и результаты
    all_day_models.append(model_day.state_dict())
    all_day_scores.append(day_results['best_score'])
    
    # Сохраняем модель дня
    torch.save(model_day.state_dict(), f"independent_day_{day_num}.pt")
    print(f"\n💾 Модель дня сохранена: independent_day_{day_num}.pt")
    print(f"📊 Лучший macro-F1 (val): {day_results['best_score']:.4f}")
    
    # ============================================================
    # УСРЕДНЕНИЕ ВЕСОВ (прогрессивное)
    # ============================================================
    
    print(f"\n{'─'*70}")
    print(f"🔗 УСРЕДНЕНИЕ ВЕСОВ: Дни 1-{day_num}")
    print(f"{'─'*70}")
    
    averaged_weights = average_weights(all_day_models)
    
    # Создаем модель с усредненными весами
    avg_model, _ = create_model(config.model)
    avg_model.load_state_dict(averaged_weights)
    avg_model = avg_model.to(config.training.device)
    
    # Сохраняем усредненную модель
    torch.save(averaged_weights, f"averaged_model_days_1_to_{day_num}.pt")
    print(f"💾 Усредненная модель сохранена: averaged_model_days_1_to_{day_num}.pt")
    
    # ============================================================
    # ТЕСТИРОВАНИЕ УСРЕДНЕННОЙ МОДЕЛИ
    # ============================================================
    
    print(f"\n{'─'*70}")
    print(f"🧪 ТЕСТ НА {config.data.test_date} (Усредненная модель из {day_num} дней)")
    print(f"{'─'*70}")
    
    test_results = evaluate_and_display(
        avg_model, 
        test_dl, 
        config.training.device,
        title=f"📈 Результаты (дни 1-{day_num})"
    )
    
    progressive_test_results.append({
        'num_days': day_num,
        'dates': config.data.train_dates[:day_num],
        **test_results
    })

print(f"\n\n{'='*70}")
print("✅ ОБУЧЕНИЕ ЗАВЕРШЕНО")
print(f"{'='*70}")

In [ ]:
# ============================================================
# ИТОГОВАЯ СВОДКА
# ============================================================

print("\n" + "="*70)
print("📊 ИТОГОВАЯ СВОДКА: ПРОГРЕССИВНОЕ УЛУЧШЕНИЕ")
print("="*70)

print("\n📋 Результаты отдельных дней (на валидации):")
print("─"*70)
for i, score in enumerate(all_day_scores):
    print(f"  День {i+1} ({config.data.train_dates[i]}): macro-F1 = {score:.4f}")

print("\n\n🎯 Прогрессивное улучшение на ТЕСТЕ ({config.data.test_date}):")
print("─"*70)
print(f"{'Дни':^15} | {'Macro F1':^10} | {'F1↓':^8} | {'F1○':^8} | {'F1↑':^8} | {'Улучшение':^12}")
print("─"*70)

for i, result in enumerate(progressive_test_results):
    improvement = ""
    if i > 0:
        prev_f1 = progressive_test_results[i-1]['macro_f1']
        curr_f1 = result['macro_f1']
        delta = (curr_f1 - prev_f1) / prev_f1 * 100
        improvement = f"{delta:+.1f}%"
    
    print(f"  1-{result['num_days']:2d}        | "
          f"{result['macro_f1']:^10.4f} | "
          f"{result['f1_down']:^8.3f} | "
          f"{result['f1_flat']:^8.3f} | "
          f"{result['f1_up']:^8.3f} | "
          f"{improvement:^12}")

print("─"*70)

# Вычисляем общее улучшение
initial_f1 = progressive_test_results[0]['macro_f1']
final_f1 = progressive_test_results[-1]['macro_f1']
total_improvement = (final_f1 - initial_f1) / initial_f1 * 100

print(f"\n📈 Общее улучшение: {initial_f1:.4f} → {final_f1:.4f} ({total_improvement:+.1f}%)")

print("\n💾 Сохраненные модели:")
print("─"*70)
for i in range(1, len(config.data.train_dates) + 1):
    print(f"  - independent_day_{i}.pt")
    print(f"  - averaged_model_days_1_to_{i}.pt")

print(f"\n🎯 Лучшая модель: averaged_model_days_1_to_{len(config.data.train_dates)}.pt")
print(f"   Macro F1 на тесте: {final_f1:.4f}")

print("\n" + "="*70)
print("✅ АНАЛИЗ ЗАВЕРШЕН")
print("="*70)

In [ ]:
# Визуализация прогресса
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# График 1: Macro F1
num_days = [r['num_days'] for r in progressive_test_results]
macro_f1s = [r['macro_f1'] for r in progressive_test_results]

axes[0].plot(num_days, macro_f1s, 'o-', linewidth=2, markersize=10, color='#2E86AB')
axes[0].set_xlabel('Количество дней в усреднении', fontsize=12)
axes[0].set_ylabel('Macro F1 Score', fontsize=12)
axes[0].set_title('📈 Прогрессивное улучшение Macro F1', fontsize=14, fontweight='bold')
axes[0].grid(True, alpha=0.3)
axes[0].set_xticks(num_days)

# График 2: F1 по классам
f1_down = [r['f1_down'] for r in progressive_test_results]
f1_flat = [r['f1_flat'] for r in progressive_test_results]
f1_up = [r['f1_up'] for r in progressive_test_results]

axes[1].plot(num_days, f1_down, 'o-', label='F1 ↓ (down)', linewidth=2, markersize=8, color='#A23B72')
axes[1].plot(num_days, f1_flat, 's-', label='F1 ○ (flat)', linewidth=2, markersize=8, color='#F18F01')
axes[1].plot(num_days, f1_up, '^-', label='F1 ↑ (up)', linewidth=2, markersize=8, color='#C73E1D')
axes[1].set_xlabel('Количество дней в усреднении', fontsize=12)
axes[1].set_ylabel('F1 Score', fontsize=12)
axes[1].set_title('📊 F1 Score по классам', fontsize=14, fontweight='bold')
axes[1].legend(loc='best', fontsize=10)
axes[1].grid(True, alpha=0.3)
axes[1].set_xticks(num_days)

plt.tight_layout()
plt.savefig('independent_training_progress.png', dpi=150, bbox_inches='tight')
plt.show()

print("📊 График сохранен: independent_training_progress.png")